# 02 - Forecasting backtest

Run `make backtest` first. Every number here comes from `ds/backtest.py` (written to
`data/reports/backtest_summary.csv` and `config/detector.json`, and logged to MLflow).
This notebook only plots them. The backtest uses 2017 only; 2018 is held back for the fault benchmark.

In [ ]:
import json
from datetime import date

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

from core.settings import REPO_ROOT, get_settings
from ds.catalogue import load_catalogue, load_detection_config
from ds.forecasting import ForecastSpec, forecast_origins, load_series, mondays

settings = get_settings()
summary = pd.read_csv(settings.reports_dir / "backtest_summary.csv")
detector = json.loads((REPO_ROOT / "config" / "detector.json").read_text(encoding="utf-8"))
detector["groups"]

## Calibrated coverage vs interval width, per series group

The chosen model per group is the narrowest one at >= 90% calibrated coverage.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6), sharey=True)
for ax, (group, g) in zip(axes.ravel(), summary.groupby("group")):
    ax.scatter(g["calibrated_scaled_width_second_half"], g["calibrated_coverage_second_half"])
    for r in g.itertuples():
        ax.annotate(r.model, (r.calibrated_scaled_width_second_half, r.calibrated_coverage_second_half), fontsize=7)
    ax.axhline(0.90, ls="--", c="grey", lw=0.8); ax.axhline(0.95, c="grey", lw=0.5)
    ax.set_title(group, fontsize=9)
axes[0, 0].set_ylabel("calibrated coverage (2nd half)"); axes[1, 0].set_xlabel("scaled interval width")
plt.tight_layout(); plt.show()
summary.pivot_table(index="group", columns="model", values="mase").round(3)

## One series: total orders, chosen model, re-fitted every Monday

In [ ]:
cfg, catalogue = load_detection_config(), load_catalogue()
con = duckdb.connect(str(settings.duckdb_path), read_only=True)
series = load_series(con, catalogue)
uid = "orders|total|all|daily"
one = series[series["unique_id"] == uid]
model = detector["groups"]["volume|daily"]["model"]
spec = ForecastSpec(detector["interval_level"], cfg.forecast.daily_season_length, cfg.forecast.weekly_season_length)
fc = forecast_origins(one, mondays(date(2017, 9, 4), date(2017, 12, 31)), {uid: [model]}, spec, cfg)
c = detector["groups"]["volume|daily"]["calibration"]
mid, half = fc["yhat"], (fc["hi"] - fc["lo"]) / 2 * c
fig, ax = plt.subplots(figsize=(12, 3.5))
actual = one[(one["ds"] >= "2017-09-04") & (one["ds"] <= "2017-12-31")]
ax.plot(actual["ds"], actual["y"], c="k", lw=0.8, label="observed")
ax.plot(fc["ds"], mid, lw=0.8, label=f"{model} forecast")
ax.fill_between(fc["ds"], mid - half, mid + half, alpha=0.2, label="calibrated 95% interval")
ax.axvspan(pd.Timestamp("2017-11-23"), pd.Timestamp("2017-11-29"), color="orange", alpha=0.15, label="Black Friday (known event)")
ax.legend(fontsize=8); ax.set_title("Total orders per day"); plt.show()